In [43]:
print("hello")

hello


In [44]:
# Solar power available during each one-hour slot, in kW.
solar = {
    9: 1.0,   # 09:00–10:00
    10: 2.0,  # 10:00–11:00
    11: 1.5,  # 11:00–12:00
}

SLOT_MINUTES = 60

# These are illustrative values, not measured appliance specifications.
appliances = {
    "washing_machine": {
        "power_kw": 0.5,
        "duration_minutes": 120,
        "earliest_start": "09:00",
        "latest_finish": "12:00",
        "preferred_start": "09:00",
        "priority": 2,
    },
    "water_pump": {
        "power_kw": 0.8,
        "duration_minutes": 60,
        "earliest_start": "09:00",
        "latest_finish": "12:00",
        "preferred_start": "09:00",
        "priority": 3,
    },
    "electric_iron": {
        "power_kw": 1.0,
        "duration_minutes": 60,
        "earliest_start": "10:00",
        "latest_finish": "12:00",
        "preferred_start": "10:00",
        "priority": 1,
    },
}

| Input | Meaning |
|---|---|
| `power_kw` | Assumed operating power while the appliance runs. |
| `duration_minutes` | Required operating duration for this job. |
| `earliest_start` | The appliance cannot start before this time. |
| `latest_finish` | The entire job must finish by this time. |
| `preferred_start` | The user’s desired start; the optimizer may move it. |
| `priority` | Higher values give more weight to preserving the preferred start. |

In [45]:
settings = {
    "solar_safety_factor": 1.0,
    "background_load_kw": 0.0,
    "max_household_power_kw": None,
}

The first complete model will enforce these rules:
1. Every appliance job runs exactly once.
2. Every job runs for its full duration, without interruption.
3. Jobs start and finish inside their allowed windows.
4. Total running power stays below available solar in every slot.
5. The optional household power limit is respected.
6. No grid power or battery is available.
7. If the requirements cannot fit, return “infeasible.”

---

## 2. find the valid starting times for each appliance

for washing machine

Start 09:00 → finish 11:00 ✓

Start 10:00 → finish 12:00 ✓

Start 11:00 → finish 13:00 ✗

In [46]:
def time_to_minutes(value):
    """Convert '09:30' into minutes after midnight (treating 00:00 (12:00 AM) as the zero-point of the day)."""
    hour, minute = map(int, value.split(":"))
    return hour * 60 + minute

def minutes_to_time(value):
    """Convert minutes after midnight into '09:30'."""
    return f"{value // 60:02d}:{value % 60:02d}"

def get_valid_starts(solar, appliances, slot_minutes):
    """
    Return valid starting slots and required slot counts.

    This checks time windows and uninterrupted slot availability.
    Solar power constraints will be added in the MILP.
    """
    available_slots = {hour * 60 for hour in solar}

    valid_starts = {}
    duration_slots = {}

    for name, appliance in appliances.items():
        duration = appliance["duration_minutes"]

        if duration <= 0 or duration % slot_minutes != 0:
            raise ValueError(
                f"{name}: duration must be a positive multiple "
                f"of {slot_minutes} minutes."
            )

        earliest = time_to_minutes(appliance["earliest_start"])
        deadline = time_to_minutes(appliance["latest_finish"])

        required_slots = duration // slot_minutes
        duration_slots[name] = required_slots

        starts = []

        for start in sorted(available_slots):
            # Check the earliest start and latest finish.
            if start < earliest or start + duration > deadline:
                continue

            # Every slot needed by the job must exist.
            occupied_slots = [
                start + offset * slot_minutes
                for offset in range(required_slots)
            ]

            if all(slot in available_slots for slot in occupied_slots):
                starts.append(start)

        valid_starts[name] = starts

    return valid_starts, duration_slots



In [47]:
# call the function

valid_starts, duration_slots = get_valid_starts(
    solar,
    appliances,
    SLOT_MINUTES,
)

for name, starts in valid_starts.items():
    readable_starts = [
        minutes_to_time(start)
        for start in starts
    ]

    print(
        f"{name}: "
        f"{duration_slots[name]} slot(s), "
        f"valid starts = {readable_starts}"
    )

washing_machine: 2 slot(s), valid starts = ['09:00', '10:00']
water_pump: 1 slot(s), valid starts = ['09:00', '10:00', '11:00']
electric_iron: 1 slot(s), valid starts = ['10:00', '11:00']


## Step 3: create the MILP model and its binary start decisions.

In [48]:
import pyomo.environ as pyo

model = pyo.ConcreteModel()

# Each pair represents one possible appliance start.
start_options = [
    (name, start)
    for name, starts in valid_starts.items()
    for start in starts
]

model.start_options = pyo.Set(
    dimen=2,
    initialize=start_options,
)

# 1 = start the appliance here.
# 0 = do not start it here.
model.start = pyo.Var(
    model.start_options,
    domain=pyo.Binary,
)

In [49]:
model.start_options.pprint()

start_options : Size=1, Index=None, Ordered=Insertion
    Key  : Dimen : Domain : Size : Members
    None :     2 :    Any :    7 : {('washing_machine', 540), ('washing_machine', 600), ('water_pump', 540), ('water_pump', 600), ('water_pump', 660), ('electric_iron', 600), ('electric_iron', 660)}


In [50]:
model.start.pprint()

start : Size=7, Index=start_options
    Key                      : Lower : Value : Upper : Fixed : Stale : Domain
      ('electric_iron', 600) :     0 :  None :     1 : False :  True : Binary
      ('electric_iron', 660) :     0 :  None :     1 : False :  True : Binary
    ('washing_machine', 540) :     0 :  None :     1 : False :  True : Binary
    ('washing_machine', 600) :     0 :  None :     1 : False :  True : Binary
         ('water_pump', 540) :     0 :  None :     1 : False :  True : Binary
         ('water_pump', 600) :     0 :  None :     1 : False :  True : Binary
         ('water_pump', 660) :     0 :  None :     1 : False :  True : Binary


In [51]:
# adding constraint: each appliance must start exactly once.

# check every appliace has at least one candidate
for name, starts in valid_starts.items():
    if not starts:
        raise ValueError(
            f"{name} cannot fit within its allowed time window."
        )


# create the constraint
model.appliances = pyo.Set(
    initialize=list(appliances) # difined appliance list
)


def start_once_rule(m, name):
    return sum(
        m.start[name, start]
        for start in valid_starts[name]
    ) == 1
# Start at 09:00 + Start at 10:00 = 1

model.start_once = pyo.Constraint(
    model.appliances,
    rule=start_once_rule,
)

In [52]:
model.start_once.pprint()

start_once : Size=3, Index=appliances, Active=True
    Key             : Lower : Body                                                                  : Upper : Active
      electric_iron :   1.0 :                   start[electric_iron,600] + start[electric_iron,660] :   1.0 :   True
    washing_machine :   1.0 :               start[washing_machine,540] + start[washing_machine,600] :   1.0 :   True
         water_pump :   1.0 : start[water_pump,540] + start[water_pump,600] + start[water_pump,660] :   1.0 :   True


## Step 4: calculate running power and enforce the solar limit.

A start decision determines how long an appliance stays on. For example, if the washing machine starts at 09:00 and needs two hours, it occupies:

09:00–10:00
10:00–11:00

In [53]:
# Convert the solar dictionary's hour keys into minutes.
solar_by_slot = {
    hour * 60: power_kw
    for hour, power_kw in solar.items()
}

model.slots = pyo.Set(
    initialize=sorted(solar_by_slot),
    ordered=True,
)

In [54]:
model.slots.pprint()

slots : Size=1, Index=None, Ordered=Insertion
    Key  : Dimen : Domain : Size : Members
    None :     1 :    Any :    3 : {540, 600, 660}


In [55]:
# Now define whether each appliance is running in each slot

def running_rule(m, name, slot):
    duration = appliances[name]["duration_minutes"]

    return sum(
        m.start[name, start]
        for start in valid_starts[name]
        if start <= slot < start + duration
    )

# condition: start <= slot < start + duration 
model.running = pyo.Expression(
    model.appliances,
    model.slots,
    rule=running_rule,
)

In [56]:
model.running.pprint()

running : Size=9, Index=appliances*slots
    Key                      : Expression
      ('electric_iron', 540) : 0.0
      ('electric_iron', 600) : start[electric_iron,600]
      ('electric_iron', 660) : start[electric_iron,660]
    ('washing_machine', 540) : start[washing_machine,540]
    ('washing_machine', 600) : start[washing_machine,540] + start[washing_machine,600]
    ('washing_machine', 660) : start[washing_machine,600]
         ('water_pump', 540) : start[water_pump,540]
         ('water_pump', 600) : start[water_pump,600]
         ('water_pump', 660) : start[water_pump,660]


includes every occupied slot and excludes the finishing boundary. A washing machine starting at 09:00 for two hours finishes at 11:00, so it does not occupy 11:00–12:00.

In [57]:
#Calculate the combined household load
def total_load_rule(m, slot):
    appliance_load = sum(
        appliances[name]["power_kw"] * m.running[name, slot]
        for name in m.appliances
    )

    return (
        settings["background_load_kw"]
        + appliance_load
    )


model.total_load = pyo.Expression(
    model.slots,
    rule=total_load_rule,
)

In [58]:
model.total_load.pprint()

total_load : Size=3, Index=slots
    Key : Expression
    540 : 0.5*start[washing_machine,540] + 0.8*start[water_pump,540] + 0
    600 : 0.5*(start[washing_machine,540] + start[washing_machine,600]) + 0.8*start[water_pump,600] + start[electric_iron,600]
    660 : 0.5*start[washing_machine,600] + 0.8*start[water_pump,660] + start[electric_iron,660]


In [59]:
# enforce the solar-only constraint
def solar_limit_rule(m, slot):
    usable_solar = (
        solar_by_slot[slot]
        * settings["solar_safety_factor"]
    )

    return m.total_load[slot] <= usable_solar


model.solar_limit = pyo.Constraint(
    model.slots,
    rule=solar_limit_rule,
)

Background load
+ power of every appliance running at 09:00
≤ available solar at 09:00

In [60]:
model.solar_limit.pprint()

solar_limit : Size=3, Index=slots, Active=True
    Key : Lower : Body            : Upper : Active
    540 :  -Inf : total_load[540] :   1.0 :   True
    600 :  -Inf : total_load[600] :   2.0 :   True
    660 :  -Inf : total_load[660] :   1.5 :   True


In [61]:
# optional household power limit
def household_limit_rule(m, slot):
    limit = settings["max_household_power_kw"]

    if limit is None:
        return pyo.Constraint.Skip

    return m.total_load[slot] <= limit


model.household_limit = pyo.Constraint(
    model.slots,
    rule=household_limit_rule,
)

In [62]:
model.household_limit.pprint()

household_limit : Size=0, Index=slots, Active=True
    Key : Lower : Body : Upper : Active


## Step 5: define what makes one feasible schedule better than another.

1. Minimize priority-weighted movement from preferred start times.
2. Among schedules equally good on that measure, minimize peak household power.

In [63]:
# calculate the penalty for each possible start
start_penalty = {}

for name, start in start_options:
    preferred = time_to_minutes(
        appliances[name]["preferred_start"]
    )

    priority = appliances[name]["priority"]

    start_penalty[name, start] = (
        abs(start - preferred) * priority
    )

Washing machine: <br>
Preferred start: 09:00<br>
Priority: 2<br>

Start at 09:00 →  0 minutes × 2 =   0<br>
Start at 10:00 → 60 minutes × 2 = 120

The absolute difference penalizes starting earlier or later than preferred.

In [64]:
# total penalty expression
model.total_start_penalty = pyo.Expression(
    expr=sum(
        start_penalty[name, start]
        * model.start[name, start]
        for name, start in model.start_options
    )
)

In [65]:
model.total_start_penalty.pprint()

total_start_penalty : Size=1, Index=None
    Key  : Expression
    None : 0*start[washing_machine,540] + 120*start[washing_machine,600] + 0*start[water_pump,540] + 180*start[water_pump,600] + 360*start[water_pump,660] + 0*start[electric_iron,600] + 60*start[electric_iron,660]


In [66]:
# define a variable for peak household power
model.peak_power = pyo.Var(
    domain=pyo.NonNegativeReals
)

In [67]:
model.peak_power.pprint()

peak_power : Size=1, Index=None
    Key  : Lower : Value : Upper : Fixed : Stale : Domain
    None :     0 :  None :  None : False :  True : NonNegativeReals


In [68]:
# It must be at least as large as the load in every slot
def peak_power_rule(m, slot):
    return m.peak_power >= m.total_load[slot]


model.peak_power_limit = pyo.Constraint(
    model.slots,
    rule=peak_power_rule,
)

In [69]:
model.peak_power_limit.pprint()

peak_power_limit : Size=3, Index=slots, Active=True
    Key : Lower : Body                                                                                                                                : Upper : Active
    540 :  -Inf :                                                       (0.5*start[washing_machine,540] + 0.8*start[water_pump,540] + 0) - peak_power :   0.0 :   True
    600 :  -Inf : (0.5*(start[washing_machine,540] + start[washing_machine,600]) + 0.8*start[water_pump,600] + start[electric_iron,600]) - peak_power :   0.0 :   True
    660 :  -Inf :                                (0.5*start[washing_machine,600] + 0.8*start[water_pump,660] + start[electric_iron,660]) - peak_power :   0.0 :   True


When minimize this variable, it becomes the highest slot load

In [70]:
model.objective = pyo.Objective(
    expr=model.total_start_penalty,
    sense=pyo.minimize,
)

In [71]:
model.objective.pprint()

objective : Size=1, Index=None, Active=True
    Key  : Active : Sense    : Expression
    None :   True : minimize : total_start_penalty


In [72]:
# Inspect the candidate penalties
for name, start in start_options:
    print(
        f"{name:20} "
        f"{minutes_to_time(start)} "
        f"penalty = {start_penalty[name, start]}"
    )

washing_machine      09:00 penalty = 0
washing_machine      10:00 penalty = 120
water_pump           09:00 penalty = 0
water_pump           10:00 penalty = 180
water_pump           11:00 penalty = 360
electric_iron        10:00 penalty = 0
electric_iron        11:00 penalty = 60


## Step 6: solve the model in two stages.

In [73]:
# find the smallest start-time penalty. 
# Then preserve that result and minimize peak power.

solver = pyo.SolverFactory("appsi_highs")

if not solver.available(exception_flag=False):
    raise RuntimeError(
        "HiGHS is unavailable. Install it with: pip install highspy"
    )

solver.options["mip_rel_gap"] = 0.0

schedule_ready = False

# Avoid automatically loading a solution when none exists.
first_result = solver.solve(
    model,
    load_solutions=False,
)

first_status = first_result.solver.termination_condition

print("Stage 1:", first_status)

Stage 1: optimal


In [74]:
# check the result and perform the second solve:
if first_status == pyo.TerminationCondition.optimal:
    solver.load_vars()

    best_penalty = pyo.value(
        model.total_start_penalty
    )

    print("Minimum weighted start penalty:", best_penalty)

    # Preserve the first objective's optimum.
    model.keep_best_start_penalty = pyo.Constraint(
        expr=model.total_start_penalty <= best_penalty + 1e-6
    )

    # Change the objective to minimizing peak power.
    model.objective.set_value(model.peak_power)

    second_result = solver.solve(
        model,
        load_solutions=False,
    )

    second_status = (
        second_result.solver.termination_condition
    )

    print("Stage 2:", second_status)

    if second_status == pyo.TerminationCondition.optimal:
        solver.load_vars()
        schedule_ready = True

        print(
            "Final weighted start penalty:",
            pyo.value(model.total_start_penalty),
        )
        print(
            "Minimum peak power:",
            pyo.value(model.peak_power),
            "kW",
        )
    else:
        print("The second-stage optimum was not found.")

elif first_status == pyo.TerminationCondition.infeasible:
    print(
        "No solar-only schedule can satisfy all requirements."
    )

else:
    print("The first-stage optimum was not found.")

Minimum weighted start penalty: 120.0
Stage 2: optimal
Final weighted start penalty: 120.0
Minimum peak power: 1.5 kW
Stage 2: optimal
Final weighted start penalty: 120.0
Minimum peak power: 1.5 kW


In [75]:
# Display the schedule only when both stages succeed
if schedule_ready:
    print("\nAppliance schedule")

    for name in model.appliances:
        chosen_start = next(
            start
            for start in valid_starts[name]
            if pyo.value(model.start[name, start]) > 0.5
        )

        finish = (
            chosen_start
            + appliances[name]["duration_minutes"]
        )

        print(
            f"{name}: "
            f"{minutes_to_time(chosen_start)}–"
            f"{minutes_to_time(finish)}"
        )

    print("\nPower by slot")

    for slot in model.slots:
        running = [
            name
            for name in model.appliances
            if pyo.value(model.running[name, slot]) > 0.5
        ]

        print(
            f"{minutes_to_time(slot)}: "
            f"{', '.join(running) or 'No appliances'} | "
            f"load = {pyo.value(model.total_load[slot]):.2f} kW | "
            f"usable solar = "
            f"{solar_by_slot[slot] * settings['solar_safety_factor']:.2f} kW"
        )


Appliance schedule
washing_machine: 10:00–12:00
water_pump: 09:00–10:00
electric_iron: 10:00–11:00

Power by slot
09:00: water_pump | load = 0.80 kW | usable solar = 1.00 kW
10:00: washing_machine, electric_iron | load = 1.50 kW | usable solar = 2.00 kW
11:00: washing_machine | load = 0.50 kW | usable solar = 1.50 kW


## Step 7: independently check the schedule.

“Optimal” means the solver optimized the model. should also check that its selected starts satisfy intended rules.

This evaluation reconstructs appliance operation from the selected starts, rather than using model.running or model.total_load.

In [76]:
# This evaluation reconstructs appliance operation from the selected starts, 
# rather than using model.running or model.total_load.

def evaluate_schedule(
    model,
    solar,
    appliances,
    settings,
    slot_minutes,
    tolerance=1e-6,
):
    solar_slots = {
        hour * 60: power
        for hour, power in solar.items()
    }

    errors = []
    selected_starts = {}
    schedule_rows = []

    # 1. Check each appliance's selected start and time window.
    for name, appliance in appliances.items():
        chosen = [
            start
            for appliance_name, start in model.start_options
            if appliance_name == name
            and pyo.value(model.start[name, start]) > 0.5
        ]

        if len(chosen) != 1:
            errors.append(
                f"{name}: expected exactly one start; "
                f"found {len(chosen)}."
            )
            continue

        start = chosen[0]
        duration = appliance["duration_minutes"]
        finish = start + duration

        earliest = time_to_minutes(appliance["earliest_start"])
        deadline = time_to_minutes(appliance["latest_finish"])

        if start < earliest:
            errors.append(f"{name}: starts too early.")

        if finish > deadline:
            errors.append(f"{name}: finishes after its deadline.")

        occupied_slots = list(
            range(start, finish, slot_minutes)
        )

        if any(slot not in solar_slots for slot in occupied_slots):
            errors.append(
                f"{name}: operation extends outside available slots."
            )

        selected_starts[name] = start

        schedule_rows.append({
            "appliance": name,
            "start": minutes_to_time(start),
            "finish": minutes_to_time(finish),
            "duration_minutes": duration,
            "energy_kwh": (
                appliance["power_kw"] * duration / 60
            ),
        })

    # 2. Reconstruct load and check every slot.
    slot_rows = []

    for slot, solar_kw in sorted(solar_slots.items()):
        running = [
            name
            for name, start in selected_starts.items()
            if start <= slot
            < start + appliances[name]["duration_minutes"]
        ]

        appliance_kw = sum(
            appliances[name]["power_kw"]
            for name in running
        )

        total_kw = (
            settings["background_load_kw"] + appliance_kw
        )

        usable_solar_kw = (
            solar_kw * settings["solar_safety_factor"]
        )

        if total_kw > usable_solar_kw + tolerance:
            errors.append(
                f"{minutes_to_time(slot)}: solar limit exceeded."
            )

        household_limit = settings["max_household_power_kw"]

        if (
            household_limit is not None
            and total_kw > household_limit + tolerance
        ):
            errors.append(
                f"{minutes_to_time(slot)}: household limit exceeded."
            )

        slot_rows.append({
            "time": minutes_to_time(slot),
            "running": ", ".join(running),
            "appliance_kw": appliance_kw,
            "total_kw": total_kw,
            "usable_solar_kw": usable_solar_kw,
        })

    # 3. Check energy using two separate calculations.
    slot_hours = slot_minutes / 60

    expected_appliance_kwh = sum(
        appliance["power_kw"]
        * appliance["duration_minutes"] / 60
        for appliance in appliances.values()
    )

    scheduled_appliance_kwh = sum(
        row["appliance_kw"] * slot_hours
        for row in slot_rows
    )

    if abs(
        scheduled_appliance_kwh - expected_appliance_kwh
    ) > tolerance:
        errors.append("Scheduled appliance energy is incorrect.")

    return {
        "valid": not errors,
        "errors": errors,
        "schedule": schedule_rows,
        "slots": slot_rows,
        "appliance_energy_kwh": scheduled_appliance_kwh,
        "peak_household_kw": max(
            row["total_kw"] for row in slot_rows
        ),
    }

In [77]:
import pandas as pd
from IPython.display import display

if schedule_ready:
    evaluation = evaluate_schedule(
        model,
        solar,
        appliances,
        settings,
        SLOT_MINUTES,
    )

    print("Valid schedule:", evaluation["valid"])
    print("Errors:", evaluation["errors"])

    display(pd.DataFrame(evaluation["schedule"]))
    display(pd.DataFrame(evaluation["slots"]))

    print(
        "Appliance energy:",
        evaluation["appliance_energy_kwh"],
        "kWh",
    )
    print(
        "Peak household power:",
        evaluation["peak_household_kw"],
        "kW",
    )

Valid schedule: True
Errors: []


,appliance,start,finish,duration_minutes,energy_kwh
0,washing_machine,10:00,12:00,120,1.0
1,water_pump,09:00,10:00,60,0.8
2,electric_iron,10:00,11:00,60,1.0


,time,running,appliance_kw,total_kw,usable_solar_kw
0,09:00,water_pump,0.8,0.8,1.0
1,10:00,"washing_machine, electric_iron",1.5,1.5,2.0
2,11:00,washing_machine,0.5,0.5,1.5


Appliance energy: 2.8 kWh
Peak household power: 1.5 kW


## Step 8: compare the optimized schedule with the preferred-time schedule.

The baseline starts every appliance at its preferred time, without coordinating solar availability.

| Slot | Appliances | Load | Solar | Result |
|---|---|---:|---:|---|
| 09:00–10:00 | Washing machine + pump | 1.3 kW | 1.0 kW | Exceeds solar |
| 10:00–11:00 | Washing machine + iron | 1.5 kW | 2.0 kW | Fits |
| 11:00–12:00 | None | 0.0 kW | 1.5 kW | Fits |

In [78]:
def summarize_starts(
    starts,
    solar,
    appliances,
    settings,
    slot_minutes,
    tolerance=1e-6,
):
    """
    Summarize a schedule represented by:
        {appliance_name: start_in_minutes}

    Intended for comparing these already-defined example schedules.
    """
    slot_hours = slot_minutes / 60
    slot_rows = []

    for hour, solar_kw in sorted(solar.items()):
        slot = hour * 60

        running = [
            name
            for name, start in starts.items()
            if start <= slot
            < start + appliances[name]["duration_minutes"]
        ]

        load_kw = settings["background_load_kw"] + sum(
            appliances[name]["power_kw"]
            for name in running
        )

        usable_solar_kw = (
            solar_kw * settings["solar_safety_factor"]
        )

        deficit_kw = max(load_kw - usable_solar_kw, 0.0)

        slot_rows.append({
            "time": minutes_to_time(slot),
            "load_kw": load_kw,
            "usable_solar_kw": usable_solar_kw,
            "solar_deficit_kw": deficit_kw,
        })

    weighted_start_penalty = sum(
        appliances[name]["priority"]
        * abs(
            start
            - time_to_minutes(
                appliances[name]["preferred_start"]
            )
        )
        for name, start in starts.items()
    )

    return {
        "peak_load_kw": max(
            row["load_kw"] for row in slot_rows
        ),
        "load_energy_kwh": sum(
            row["load_kw"] * slot_hours
            for row in slot_rows
        ),
        "solar_violating_slots": sum(
            row["solar_deficit_kw"] > tolerance
            for row in slot_rows
        ),
        "solar_deficit_kwh": sum(
            row["solar_deficit_kw"] * slot_hours
            for row in slot_rows
        ),
        "weighted_start_penalty": weighted_start_penalty,
        "slots": slot_rows,
    }

In [79]:
# Extract the two schedules and compare them
if schedule_ready:
    baseline_starts = {
        name: time_to_minutes(appliance["preferred_start"])
        for name, appliance in appliances.items()
    }

    optimized_starts = {
        name: next(
            start
            for start in valid_starts[name]
            if pyo.value(model.start[name, start]) > 0.5
        )
        for name in appliances
    }

    baseline = summarize_starts(
        baseline_starts,
        solar,
        appliances,
        settings,
        SLOT_MINUTES,
    )

    optimized = summarize_starts(
        optimized_starts,
        solar,
        appliances,
        settings,
        SLOT_MINUTES,
    )

    comparison = pd.DataFrame({
        "Preferred-time baseline": {
            key: value
            for key, value in baseline.items()
            if key != "slots"
        },
        "Optimized": {
            key: value
            for key, value in optimized.items()
            if key != "slots"
        },
    })

    display(comparison)

,Preferred-time baseline,Optimized
peak_load_kw,1.5,1.5
load_energy_kwh,2.8,2.8
solar_violating_slots,1.0,0.0
solar_deficit_kwh,0.3,0.0
weighted_start_penalty,0.0,120.0


**Yes—for demonstrating that the algorithm works. This small example alone is not enough to establish research effectiveness.**

You can present it as an **illustrative case study** showing that the optimizer:

- Completes every appliance job.
- Respects durations, deadlines, and solar limits.
- Removes the baseline’s **0.3 kWh solar shortfall** by shifting the washing machine.

Clearly state that the appliance powers, requirements, and solar values are **assumed example inputs**. Don’t describe the result as measured Sri Lankan household performance.

For your main research evaluation, we should extend it:

| Evaluation | What it establishes |
|---|---|
| Multiple days with different solar conditions | Performance beyond one convenient example |
| Different appliance combinations and time windows | Ability to handle varied household requirements |
| Preferred-time and simple heuristic baselines | Whether MILP improves on simpler approaches |
| Constraint checks and infeasible cases | Correctness and honest handling of impossible requests |
| Solve time and optimality status | Practical suitability for backend use |
| Scheduling with predicted solar, checked against actual solar | Whether schedules remain feasible when forecasts are imperfect |

Report **job completion, solar-limit violations, unmet energy, weighted start-time deviation, peak load, and runtime**. Compare convenience scores only between schedules that complete the same required jobs and satisfy the constraints.

Also, in this example:

- Appliance energy stays **2.8 kWh**.
- Peak power stays **1.5 kW**.
- The improvement is **eliminating the solar shortfall**, not reducing energy consumption.

A suitable description is:

> “An illustrative three-appliance scenario was used to verify the scheduling formulation. The optimized schedule completed all required jobs and eliminated the baseline solar-energy deficit of 0.3 kWh.”

We can keep this as the **first correctness demonstration**, then build a broader experiment for the research results.

## Step 9: call everything through one reusable function

funtion process

Validate inputs → Build model → Solve both objectives
                → Check schedule → Return results

In [80]:
"""Step 9: reusable version of the three-appliance, hourly solar-only MILP.

This keeps the tutorial's solar={9: 1.0, ...} input format. Subhourly timestamp
inputs will be introduced separately; no tariff, grid, battery, or API code.
"""
import math
from time import perf_counter

import pyomo.environ as pyo



def time_to_minutes(value):
    hour, minute = map(int, value.split(":"))
    if not 0 <= hour <= 23 or not 0 <= minute <= 59:
        raise ValueError(f"Invalid time: {value}")
    return hour * 60 + minute


def minutes_to_time(value):
    return f"{value // 60:02d}:{value % 60:02d}"


def _evaluate(starts, solar_slots, appliances, settings, slot_minutes):
    """Reconstruct operation from starts, independently of model expressions."""
    errors, schedule_rows, slot_rows = [], [], []
    penalty = 0
    for name, appliance in appliances.items():
        start = starts[name]
        duration = appliance["duration_minutes"]
        finish = start + duration
        if start < time_to_minutes(appliance["earliest_start"]):
            errors.append(f"{name}: starts too early.")
        if finish > time_to_minutes(appliance["latest_finish"]):
            errors.append(f"{name}: finishes too late.")
        if any(slot not in solar_slots for slot in range(start, finish, slot_minutes)):
            errors.append(f"{name}: required slots are missing.")
        penalty += appliance["priority"] * abs(start - time_to_minutes(appliance["preferred_start"]))
        schedule_rows.append({
            "appliance": name, "start": minutes_to_time(start),
            "finish": minutes_to_time(finish), "power_kw": appliance["power_kw"],
            "duration_minutes": duration, "energy_kwh": appliance["power_kw"] * duration / 60,
        })
    for slot, solar_kw in sorted(solar_slots.items()):
        running = [name for name, start in starts.items() if start <= slot < start + appliances[name]["duration_minutes"]]
        appliance_kw = sum(appliances[name]["power_kw"] for name in running)
        total_kw = settings["background_load_kw"] + appliance_kw
        usable_kw = solar_kw * settings["solar_safety_factor"]
        if total_kw > usable_kw + 1e-6:
            errors.append(f"{minutes_to_time(slot)}: solar limit exceeded.")
        limit = settings["max_household_power_kw"]
        if limit is not None and total_kw > limit + 1e-6:
            errors.append(f"{minutes_to_time(slot)}: household limit exceeded.")
        slot_rows.append({"time": minutes_to_time(slot), "running": running,
                          "appliance_kw": appliance_kw, "total_kw": total_kw,
                          "usable_solar_kw": usable_kw})
    energy = sum(row["appliance_kw"] * slot_minutes / 60 for row in slot_rows)
    expected = sum(row["energy_kwh"] for row in schedule_rows)
    if abs(energy - expected) > 1e-6:
        errors.append("Scheduled energy does not match required job energy.")
    return {
        "valid": not errors, "errors": errors, "schedule": schedule_rows,
        "slots": slot_rows, "appliance_energy_kwh": energy,
        "peak_household_kw": max(row["total_kw"] for row in slot_rows),
        "weighted_start_penalty": penalty,
    }


def optimize_appliance_schedule(solar, appliances, settings=None, slot_minutes=60):
    """Validate -> build -> solve twice -> independently evaluate -> return.

    Each call creates a fresh model. Missing solver/invalid inputs raise errors;
    valid but impossible scheduling requests return status='infeasible'.
    Results contain only plain Python values, suitable for later JSON responses.
    """
    started = perf_counter()
    settings = {
        "solar_safety_factor": 1.0,
        "background_load_kw": 0.0,
        "max_household_power_kw": None,
        **(settings or {}),
    }
    if slot_minutes != 60:
        raise ValueError("This tutorial step uses hourly solar keys. Use slot_minutes=60.")
    if not solar or not appliances:
        raise ValueError("Provide solar data and at least one appliance.")
    if any(not isinstance(hour, int) or not 0 <= hour <= 23 for hour in solar):
        raise ValueError("Solar keys must be integer hours from 0 through 23.")
    if any(not math.isfinite(power) or power < 0 for power in solar.values()):
        raise ValueError("Solar power must be finite and nonnegative.")
    factor, background, limit = (settings[key] for key in ("solar_safety_factor", "background_load_kw", "max_household_power_kw"))
    if not math.isfinite(factor) or not 0 < factor <= 1:
        raise ValueError("solar_safety_factor must be in (0, 1].")
    if not math.isfinite(background) or background < 0:
        raise ValueError("background_load_kw must be finite and nonnegative.")
    if limit is not None and (not math.isfinite(limit) or limit <= 0):
        raise ValueError("max_household_power_kw must be positive or None.")

    solar_slots = {hour * 60: power for hour, power in solar.items()}
    valid_starts = {}
    for name, appliance in appliances.items():
        power, duration = appliance["power_kw"], appliance["duration_minutes"]
        priority = appliance["priority"]
        if not math.isfinite(power) or power <= 0:
            raise ValueError(f"{name}: power_kw must be finite and positive.")
        if not isinstance(duration, int) or duration <= 0 or duration % slot_minutes:
            raise ValueError(f"{name}: duration must be a positive multiple of {slot_minutes}.")
        if not isinstance(priority, int) or priority < 1:
            raise ValueError(f"{name}: priority must be a positive integer.")
        earliest, deadline, preferred = [time_to_minutes(appliance[key]) for key in ("earliest_start", "latest_finish", "preferred_start")]
        if earliest >= deadline:
            raise ValueError(f"{name}: earliest_start must be before latest_finish.")
        valid_starts[name] = [start for start in sorted(solar_slots)
            if start >= earliest and start + duration <= deadline
            and all(slot in solar_slots for slot in range(start, start + duration, slot_minutes))]
        if not valid_starts[name]:
            return {"status": "infeasible", "reason": f"{name}: no start fits its full duration and time window.", "evaluation": None, "solve_seconds": perf_counter() - started}

    # Steps 3–5: start decisions, complete jobs, power limits, and objectives.
    model = pyo.ConcreteModel()
    options = [(name, start) for name, starts in valid_starts.items() for start in starts]
    model.start_options = pyo.Set(dimen=2, initialize=options)
    model.start = pyo.Var(model.start_options, domain=pyo.Binary)
    model.appliances = pyo.Set(initialize=list(appliances))
    model.slots = pyo.Set(initialize=sorted(solar_slots), ordered=True)
    model.start_once = pyo.Constraint(model.appliances, rule=lambda m, name:
        sum(m.start[name, start] for start in valid_starts[name]) == 1)
    model.running = pyo.Expression(model.appliances, model.slots, rule=lambda m, name, slot:
        sum(m.start[name, start] for start in valid_starts[name]
            if start <= slot < start + appliances[name]["duration_minutes"]))
    model.total_load = pyo.Expression(model.slots, rule=lambda m, slot:
        background + sum(appliances[name]["power_kw"] * m.running[name, slot] for name in m.appliances))
    model.solar_limit = pyo.Constraint(model.slots, rule=lambda m, slot:
        m.total_load[slot] <= solar_slots[slot] * factor)
    if limit is not None:
        model.household_limit = pyo.Constraint(model.slots, rule=lambda m, slot: m.total_load[slot] <= limit)
    model.peak_power = pyo.Var(domain=pyo.NonNegativeReals)
    model.peak_limit = pyo.Constraint(model.slots, rule=lambda m, slot: m.peak_power >= m.total_load[slot])
    model.total_start_penalty = pyo.Expression(expr=sum(
        appliances[name]["priority"] * abs(start - time_to_minutes(appliances[name]["preferred_start"])) * model.start[name, start]
        for name, start in options))
    model.objective = pyo.Objective(expr=model.total_start_penalty, sense=pyo.minimize)

    # Step 6: solve priorities first, then peak power while preserving priorities.
    solver = pyo.SolverFactory("appsi_highs")
    if not solver.available(exception_flag=False):
        raise RuntimeError("Install the HiGHS solver: pip install highspy")
    solver.options["mip_rel_gap"] = 0.0
    # solver.options["threads"] = 1
    solver.options["time_limit"] = 30.0
    for stage in (1, 2):
        result = solver.solve(model, load_solutions=False)
        condition = result.solver.termination_condition
        if condition != pyo.TerminationCondition.optimal:
            return {"status": "infeasible" if condition == pyo.TerminationCondition.infeasible else "not_proven_optimal",
                    "reason": f"Stage {stage}: {condition}", "evaluation": None,
                    "solve_seconds": perf_counter() - started}
        solver.load_vars()
        if stage == 1:
            best_penalty = float(pyo.value(model.total_start_penalty))
            model.keep_best_start_penalty = pyo.Constraint(expr=model.total_start_penalty <= best_penalty + 1e-6)
            model.objective.set_value(model.peak_power)

    starts = {}
    for name, candidates in valid_starts.items():
        chosen = [start for start in candidates if pyo.value(model.start[name, start]) > .5]
        if len(chosen) != 1:
            raise RuntimeError(f"{name}: solver did not select exactly one start.")
        starts[name] = chosen[0]
    evaluation = _evaluate(starts, solar_slots, appliances, settings, slot_minutes)
    if not evaluation["valid"]:
        raise RuntimeError(f"Independent validation failed: {evaluation['errors']}")
    return {"status": "optimal", "evaluation": evaluation, "solve_seconds": perf_counter() - started}


In [81]:
result = optimize_appliance_schedule(
    solar=solar,
    appliances=appliances,
    settings=settings,
    slot_minutes=60,
)

In [82]:
import pandas as pd

print("Status:", result["status"])

if result["status"] == "optimal":
    evaluation = result["evaluation"]

    display(pd.DataFrame(evaluation["schedule"]))
    display(pd.DataFrame(evaluation["slots"]))

    print("Valid:", evaluation["valid"])
    print("Energy:", evaluation["appliance_energy_kwh"], "kWh")
    print("Peak power:", evaluation["peak_household_kw"], "kW")
    print(
        "Weighted start penalty:",
        evaluation["weighted_start_penalty"],
    )

else:
    print(result["reason"])

Status: optimal


,appliance,start,finish,power_kw,duration_minutes,energy_kwh
0,washing_machine,10:00,12:00,0.5,120,1.0
1,water_pump,09:00,10:00,0.8,60,0.8
2,electric_iron,10:00,11:00,1.0,60,1.0


,time,running,appliance_kw,total_kw,usable_solar_kw
0,09:00,[water_pump],0.8,0.8,1.0
1,10:00,"[washing_machine, electric_iron]",1.5,1.5,2.0
2,11:00,[washing_machine],0.5,0.5,1.5


Valid: True
Energy: 2.8 kWh
Peak power: 1.5 kW
Weighted start penalty: 120


In [83]:
low_solar = {
    9: 0.1,
    10: 0.1,
    11: 0.1,
}

result = optimize_appliance_schedule(
    solar=low_solar,
    appliances=appliances,
    settings=settings,
)

print(result["status"])  # infeasible

infeasible
